# Часть 1. Проверка гипотезы в Python и составление аналитической записки

## Проверка гипотез

## Цели и задачи проекта
Цель - Провести проверку гипотез, оценить результаты, составить аналитическую записку на основе полученных данных. 

Нулевая гипотеза $H_0: \mu_{\text{СПб}} \leq \mu_{\text{Москва}}$ <br> Среднее время активности пользователей в Санкт-Петербурге не больше, чем в Москве.

Альтернативная гипотеза $H_1: \mu_{\text{СПб}} > \mu_{\text{Москва}}$ <br> Среднее время активности пользователей в Санкт-Петербурге больше, и это различие статистически значимо.

## Описание данных

```city``` - город, в котором проводилось тестирование    
```puid``` - уникальный id пользователя     
```hours``` - количество часов прослушивания книг пользователем      

## Содержимое проекта

1. Проверить наличие дубликатов
2. Сравнить размеры выборочных групп
3. Проверить распределение данных
4. Проверить статистики

## 1. Загрузка и проверка данных

In [2]:
# Импортируем билиотеки
import pandas as pd
from scipy import stats

In [ ]:
# Загрузим данные
df = pd.read_csv('секретная_информация/yandex_knigi_data.csv')

In [4]:
# Проверим вывод первых 5 строк, получим краткую информацию о данных
display(df.head())
display(df.info())

,Unnamed: 0,city,puid,hours
0,0,Москва,9668,26.167776
1,1,Москва,16598,82.111217
2,2,Москва,80401,4.656906
3,3,Москва,140205,1.840556
4,4,Москва,248755,151.326434


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8784 entries, 0 to 8783
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Unnamed: 0  8784 non-null   int64  
 1   city        8784 non-null   object 
 2   puid        8784 non-null   int64  
 3   hours       8784 non-null   float64
dtypes: float64(1), int64(2), object(1)
memory usage: 274.6+ KB


None

In [5]:
# Проверим наличие дубликатов
display(df.duplicated().sum())
display(df.duplicated('puid').sum())

np.int64(0)

np.int64(244)

In [8]:
# Удалим дубликаты по полю puid
df.drop_duplicates('puid', keep='first', inplace=True)

# Проверим изменения
display(df.duplicated('puid').sum())

np.int64(0)

In [9]:
# Выведем общую информацию после изменений
display(df.info())

<class 'pandas.core.frame.DataFrame'>
Index: 8540 entries, 0 to 8783
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Unnamed: 0  8540 non-null   int64  
 1   city        8540 non-null   object 
 2   puid        8540 non-null   int64  
 3   hours       8540 non-null   float64
dtypes: float64(1), int64(2), object(1)
memory usage: 333.6+ KB


None

По результату проверки данных выявлено:
- отсутствие пропусков
- наличие 244 дубликатов по полю "puid"
Дубликаты удалены, данные готовы для тестирования.

## 2. Проверка гипотезы

Гипотеза звучит так: пользователи из Санкт-Петербурга проводят в среднем больше времени за чтением и прослушиванием книг в приложении, чем пользователи из Москвы. 

- Нулевая гипотеза H₀: Средняя активность пользователей в часах в двух группах (Москва и Санкт-Петербург) не различается.

- Альтернативная гипотеза H₁: Средняя активность пользователей в Санкт-Петербурге больше, и это различие статистически значимо.

In [10]:
# Разделим датафрейм на группы 
df_moscow = df[df['city'] == 'Москва']
df_piter = df[df['city'] == 'Санкт-Петербург']

In [11]:
# Сравним размеры выборок
share_size_group = pd.DataFrame({
    'Москва': df_moscow.count(), 
    'Санкт-Петербург': df_piter.count(), 
    'Разница (%)': round(abs((df_piter.count() - df_moscow.count()) / df_moscow.count()) * 100, 2)
                         })
display(share_size_group.head())

,Москва,Санкт-Петербург,Разница (%)
Unnamed: 0,6234,2306,63.01
city,6234,2306,63.01
puid,6234,2306,63.01
hours,6234,2306,63.01


In [12]:
# Проверим наличие пересечений уникальных пользователей в группах
not_unique = set(df_moscow['puid']).intersection(df_piter['puid'])

display(list(not_unique))

[]

In [13]:
# Используем t-тест Уэлча, т.к. размеры наблюдений >30, а нормальность распределения дисперсии неизвестна
stat_wtt, p_value_wtt = stats.ttest_ind(
    df_piter.hours,
    df_moscow.hours,
    equal_var=False,
    alternative='greater'
)
alpha = 0.05

# H0 = Uсбп <= Uмск , Н1 = Uспб > Uмск
if p_value_wtt > alpha:
    print(f'P-value Уэлча = {round(p_value_wtt, 5)}\nНулевая гипотеза подтверждена!')
else:
    print(f'p-value Уэлча = {round(p_value_wtt, 5)}\nНулевая гипотеза не подтверждена!')

P-value Уэлча = 0.34357
Нулевая гипотеза подтверждена!


## 3. Аналитическая записка

Результаты тестирования и выводы:

- Для тестирования результатов наблюдений был выбран t-тест Уэлча, который подтвердил нулевую гипотезу, а именно то,    
что среднее время прослушивания книг в Москве и Санкт-Петербурге не различается.

- P-value равен 0,34357

- Можно предположить, что на результаты тестирования повлияла разница размеров выборок. Количество наблюдений для Москвы на 63% больше,    
чем для Санкт-Петербурга. 


----

# Часть 2. Анализ результатов A/B-тестирования

Задача — провести оценку результатов A/B-теста

ТЗ:   
Гипотеза заключается в следующем: упрощение интерфейса приведёт к тому, что в течение семи дней после регистрации    
в системе конверсия зарегистрированных пользователей в покупателей увеличится как минимум на три процентных пункта.

## Описание данных

Параметры теста:
- название теста: `interface_eu_test`;
- группы: А (контрольная), B (новый интерфейс).

Данные:   

`ab_test_participants.csv` — таблица участников тестов.   

  Структура файла:
- `user_id` — идентификатор пользователя;
- `group` — группа пользователя;
- `ab_test` — название теста;
- `device` — устройство, с которого происходила регистрация.

`ab_test_events.zip` — архив с одним csv-файлом, в котором собраны события 2020 года    

  Структура файла:
- `user_id` — идентификатор пользователя;
- `event_dt` — дата и время события;   
- `event_name` — тип события;
- `details` — дополнительные данные о событии.

## 1. Цели исследования.

- провести анализ данных, проверить целостность, корректность размеров выборок
- оценить корректность првоедения теста
- првоести оценку результатов А/В тестирования
- сделать вывод на основе полученных результатов

## 2. Проверка данных


In [ ]:
# Загрузим данные
participants = pd.read_csv('секретная_информация/ab_test_participants.csv')
events = pd.read_csv('секретная_информация/ab_test_events.zip',
                     parse_dates=['event_dt'], low_memory=False)

In [15]:
# Выведем первые 5 строк каждого датафрейма, общую информацио о содержимом
display(participants.head(), participants.info())
display('*------------------------*')
display(events.head(), events.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14525 entries, 0 to 14524
Data columns (total 4 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   user_id  14525 non-null  object
 1   group    14525 non-null  object
 2   ab_test  14525 non-null  object
 3   device   14525 non-null  object
dtypes: object(4)
memory usage: 454.0+ KB


,user_id,group,ab_test,device
0,0002CE61FF2C4011,B,interface_eu_test,Mac
1,001064FEAAB631A1,B,recommender_system_test,Android
2,001064FEAAB631A1,A,interface_eu_test,Android
3,0010A1C096941592,A,recommender_system_test,Android
4,001E72F50D1C48FA,A,interface_eu_test,Mac


None

'*------------------------*'

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 787286 entries, 0 to 787285
Data columns (total 4 columns):
 #   Column      Non-Null Count   Dtype         
---  ------      --------------   -----         
 0   user_id     787286 non-null  object        
 1   event_dt    787286 non-null  datetime64[ns]
 2   event_name  787286 non-null  object        
 3   details     249022 non-null  object        
dtypes: datetime64[ns](1), object(3)
memory usage: 24.0+ MB


,user_id,event_dt,event_name,details
0,GLOBAL,2020-12-01 00:00:00,End of Black Friday Ads Campaign,ZONE_CODE15
1,CCBE9E7E99F94A08,2020-12-01 00:00:11,registration,0.0
2,GLOBAL,2020-12-01 00:00:25,product_page,NaN
3,CCBE9E7E99F94A08,2020-12-01 00:00:33,login,NaN
4,CCBE9E7E99F94A08,2020-12-01 00:00:52,product_page,NaN


None

## 3. По таблице `ab_test_participants` оценим корректность проведения теста:

   3\.1 Выделим пользователей, участвующих в тесте, и проверим:

   - соответствие требованиям технического задания,

   - равномерность распределения пользователей по группам теста,

   - отсутствие пересечений с конкурирующим тестом (нет пользователей, участвующих одновременно в двух тестовых группах).

#### Проверим данные на соответствие требованиям ТЗ

In [16]:
# По требования ТЗ файл должен содержать данные о пользователях, разбитых на 2 группы
# Проверим уникальные значения поля "group", "ab_test"
display(list(participants['group'].unique()))
display(list(participants['ab_test'].unique()))


['B', 'A']

['interface_eu_test', 'recommender_system_test']

Так как в файле находятся данные для двух разных А/В тестов, уберем из файла лишние строки.

In [17]:
# Переопределим датафрейм, оставив строки для теста "interface_eu_test"
participants = participants[participants['ab_test'] == 'interface_eu_test']

# Проверим изменения
display(list(participants['ab_test'].unique()))

['interface_eu_test']

In [18]:
# Для удобства создадим два датафрейма, разбив исходный по тестовым группам
participants_group_a = participants[participants['group'] == 'A']
participants_group_b = participants[participants['group'] == 'B']

In [19]:
# Проверим раномерность распределения пользователей по группам
even_distribution = pd.DataFrame({
    'group_a': participants_group_a.count(),
    'group_b': participants_group_b.count(),
    'share': abs(round((participants_group_a.count() - participants_group_b.count()) / participants_group_b.count() * 100, 2))
})

display(even_distribution.head())

,group_a,group_b,share
user_id,5383,5467,1.54
group,5383,5467,1.54
ab_test,5383,5467,1.54
device,5383,5467,1.54


In [20]:
# Проверим пересечения групп
display(list(set(participants_group_a['user_id']).intersection(set(participants_group_b['user_id']))))

[]

In [21]:
# Проверим, участвовали ли пользователи из теста 'interface_eu_test' в тесте 'recommender_system_test'
display('Пересечение группы А "interface_eu_test" с тестом "recommender_system_test":', 
       list(set(participants_group_a['user_id'])\
                    .intersection(set(participants[participants['ab_test'] == 'recommender_system_test']['user_id']))))

display('Пересечение группы B "interface_eu_test" с тестом "recommender_system_test":', 
       list(set(participants_group_b['user_id'])\
                    .intersection(set(participants[participants['ab_test'] == 'recommender_system_test']['user_id']))))

'Пересечение группы А "interface_eu_test" с тестом "recommender_system_test":'

[]

'Пересечение группы B "interface_eu_test" с тестом "recommender_system_test":'

[]

3\.2 Проанализируем данные о пользовательской активности по таблице `ab_test_events`

In [22]:
# Проверим данные в датафрейме "events", оставим только те события, которые принадлежат пользователям проводимого тестирования
filtered_events_df = events[events['user_id'].isin(participants['user_id'])].reset_index(drop = True)

display(filtered_events_df.head(), filtered_events_df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 79715 entries, 0 to 79714
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   user_id     79715 non-null  object        
 1   event_dt    79715 non-null  datetime64[ns]
 2   event_name  79715 non-null  object        
 3   details     21075 non-null  object        
dtypes: datetime64[ns](1), object(3)
memory usage: 2.4+ MB


,user_id,event_dt,event_name,details
0,5F506CEBEDC05D30,2020-12-06 14:10:01,registration,0.0
1,51278A006E918D97,2020-12-06 14:37:25,registration,-3.8
2,A0C1E8EFAD874D8B,2020-12-06 17:20:22,registration,-3.32
3,275A8D6254ACF530,2020-12-06 19:36:54,registration,-0.48
4,0B704EB2DC7FCA4B,2020-12-06 19:42:20,registration,0.0


None

In [23]:
# Создадим новый датафрейм, с дайтой регистрации пользователя
df_reg = filtered_events_df[filtered_events_df.event_name == 'registration'].groupby('user_id')['event_dt'].min().reset_index()
df_reg.columns = ['user_id', 'reg_dt']
display(df_reg)

,user_id,reg_dt
0,0002CE61FF2C4011,2020-12-07 04:37:31
1,001064FEAAB631A1,2020-12-20 14:12:45
2,001E72F50D1C48FA,2020-12-17 15:44:05
3,002412F1EB3F6E38,2020-12-09 09:36:50
4,002540BE89C930FB,2020-12-08 18:06:07
...,...,...
10845,FFE600EEC4BA7685,2020-12-13 21:43:31
10846,FFE7FC140521F5F6,2020-12-23 09:10:16
10847,FFEFC0E55C1CCD4F,2020-12-13 23:52:15
10848,FFF28D02B1EACBE1,2020-12-16 08:23:56


In [24]:
# Присоединим дату регистрации к отфильрованному датафрейму
filtered_events_df_reg =  filtered_events_df.merge(df_reg, on='user_id', how='inner')

In [25]:
# В новом столбце посчитаем количество дней нового события с момента регистрации
filtered_events_df_reg['lifetime_days'] = (filtered_events_df_reg.event_dt - filtered_events_df_reg.reg_dt).dt.days

# Оставим только те события, которые произошли не позднее 7 дней с момента регистрации
filtered_events_df_reg = filtered_events_df_reg[filtered_events_df_reg.lifetime_days <= 6] 

display(filtered_events_df_reg)

,user_id,event_dt,event_name,details,reg_dt,lifetime_days
0,5F506CEBEDC05D30,2020-12-06 14:10:01,registration,0.0,2020-12-06 14:10:01,0
1,51278A006E918D97,2020-12-06 14:37:25,registration,-3.8,2020-12-06 14:37:25,0
2,A0C1E8EFAD874D8B,2020-12-06 17:20:22,registration,-3.32,2020-12-06 17:20:22,0
3,275A8D6254ACF530,2020-12-06 19:36:54,registration,-0.48,2020-12-06 19:36:54,0
4,0B704EB2DC7FCA4B,2020-12-06 19:42:20,registration,0.0,2020-12-06 19:42:20,0
...,...,...,...,...,...,...
79551,E89AF4EFC757D283,2020-12-29 21:46:43,product_cart,NaN,2020-12-23 09:35:48,6
79554,E89AF4EFC757D283,2020-12-29 21:47:56,product_cart,NaN,2020-12-23 09:35:48,6
79628,A6AFDC94A0D3B23D,2020-12-29 22:47:00,product_page,NaN,2020-12-23 13:53:33,6
79634,A6AFDC94A0D3B23D,2020-12-29 22:48:46,product_page,NaN,2020-12-23 13:53:33,6


In [26]:
# Првоедем оценку достаточности выборки для получения статистически значимых результатов А/В теста.
# Базовый показатель конверсии - 30%
# Мощность теста - 80%
# Достоверность теста - 95%
# MDE - 3%

from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import proportion_effectsize
import numpy as np

alpha = 0.05  
power = 0.8
beta = 1 - power
p1 = 0.3
mde = 0.03
p2 = p1 + mde

power_analysis = NormalIndPower()
effect_size = (p2 - p1) / np.sqrt((p1 * (1 - p1) + p2 * (1 - p2)) / 2)

sample_size = power_analysis.solve_power(
    effect_size = effect_size,
    power = power,
    alpha = alpha,
    ratio = 1
)

print(f"Необходимый размер выборки для каждой группы: {int(sample_size)}")

Необходимый размер выборки для каждой группы: 3759


#### Расчитаем для каждой группы количество посетителей, сделавших покупку, и общее количество посетителей.

In [27]:
# Присоединим к каждой группе теста таблицу с событиями
events_group_a = participants_group_a.merge(filtered_events_df_reg, on='user_id', how='left')
events_group_b = participants_group_b.merge(filtered_events_df_reg, on='user_id', how='left')

In [28]:
# Для каждой группы пасчитаем количество пользователей, совершивших покупку, и общее количество пользователей
general_stats_a = pd.DataFrame({
    'purchase': events_group_a[events_group_a.event_name == 'purchase']['user_id'].nunique(),
    'total': events_group_a['user_id'].nunique(),
    'share': round(events_group_a[events_group_a.event_name == 'purchase']['user_id']\
                   .nunique() / events_group_a['user_id'].nunique(), 4)
}, index=['Group A'])

# Для группы B
general_stats_b = pd.DataFrame({
    'purchase': events_group_b[events_group_b.event_name == 'purchase']['user_id'].nunique(),
    'total': events_group_b['user_id'].nunique(),
    'share': round(events_group_b[events_group_b.event_name == 'purchase']['user_id']\
                   .nunique() / events_group_b['user_id'].nunique(), 4)
}, index=['Group B'])

# Объединяем
general_stats = pd.concat([general_stats_a, general_stats_b])

display(general_stats)

,purchase,total,share
Group A,1480,5383,0.2749
Group B,1600,5467,0.2927


Предварительный вывод:
- конверсия в покупку, в группе В, увеличилась на 2п.п.

## 4. Оценка результатов A/B-тестирования:

In [ ]:
# Исходя из размера наблюдений и целевой метрики, проверим результаты А/В теста z - тестом пропорций
# H0: показатель конверсии не изменился (A>=B)
# H1: показатель конверсии вырос минимум на 3 п.п. (А<B)
from statsmodels.stats.proportion import proportions_ztest

stats_ztest, p_value_ztest = proportions_ztest(
    [general_stats_a.purchase.iloc[0], general_stats_b.purchase.iloc[0]],
    [general_stats_a.total.iloc[0], general_stats_b.total.iloc[0]],
    alternative='smaller'
)

alpha = 0.05

if p_value_ztest > alpha:
    print(f'pvalue={p_value_ztest} > {alpha}')
    print('Нулевая гипотеза находит подтверждение!')
else:
    print(f'pvalue={p_value_ztest} < {alpha}')
    print('Нулевая гипотеза не находит подтверждения!')

pvalue=0.02030699398306548 < 0.05
Нулевая гипотеза не находит подтверждения!


### Общий вывод
Была проведена оценка результатов А/В тестирования, данные корректны, независимы, отсутствуют дубликаты. 
Был использован Z-тест пропорций, т.к. целевая метрика - конверсия в покупку. Гипотеза о том, что конверсия увеличилась статистически значимо, была подтверждена,    
что было так же предварительно проверено по расчетам пользователей (конверсия тестовой группы увеличалсь на ~2п.п.).    
Исходя из результатов тестирования, итоговый вывод - новый дизайн сайта увеличил конверсию пользователей в покупку, изменения статистически значимы.